<html><div style="font-size:7pt">This notebook may contain text, code and images generated by artificial intelligence. Used model: claude-sonnet-4-20250514, vision model: claude-sonnet-4-20250514, endpoint: None, bia-bob version: 0.34.3.. It is good scientific practice to check the code and results it produces carefully. <a href="https://github.com/haesleinhuepf/bia-bob">Read more about code generation using bia-bob</a></div></html>

# ImageNet Image Embeddings and UMAP Visualization

This notebook demonstrates how to:
1. Download 1000 random images from the [MNist dataset](https://huggingface.co/datasets/ylecun/mnist)
2. Generate vision embeddings using the [nomic-ai/nomic-embed-vision-v1.5 model](https://huggingface.co/nomic-ai/nomic-embed-vision-v1.5)
3. Create a 3D UMAP visualization of the embeddings
4. Save the results for VEST

## Step 1: Import Required Libraries

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from datasets import load_dataset
from umap import UMAP
import random
from pathlib import Path
import requests
from io import BytesIO

print("Libraries imported successfully")

Libraries imported successfully


## Step 2: Create Directory Structure and Download ImageNet Images

In [2]:
# Create the directory structure
base_dir = Path("./")
images_dir = base_dir / "images"
images_dir.mkdir(parents=True, exist_ok=True)

# Load ImageNet dataset from HuggingFace
print("Loading ylecun/mnist dataset...")
dataset = load_dataset("ylecun/mnist", split="train", streaming=True, trust_remote_code=True)

# Download 1000 random images
downloaded_count = 0
target_count = 1000
dataset_iter = iter(dataset)

print(f"Downloading {target_count} images...")

while downloaded_count < target_count:
    try:
        sample = next(dataset_iter)
        image = sample['image']
        
        # Convert to RGB if necessary
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        # Save image
        filename = f"mnist_{downloaded_count:04d}.png"
        image_path = images_dir / filename
        image.save(image_path)
        
        downloaded_count += 1
        
        if downloaded_count % 100 == 0:
            print(f"Downloaded {downloaded_count}/{target_count} images")
            
    except Exception as e:
        print(f"Error downloading image {downloaded_count}: {e}")
        continue

print(f"Successfully downloaded {downloaded_count} images to {images_dir}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ylecun/mnist' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading ylecun/mnist dataset...
Downloaded 100/1000 images
Downloaded 200/1000 images
Downloaded 300/1000 images
Downloaded 400/1000 images
Downloaded 500/1000 images
Downloaded 600/1000 images
Downloaded 700/1000 images
Downloaded 800/1000 images
Downloaded 900/1000 images
Downloaded 1000/1000 images
Successfully downloaded 1000 images to images


## Step 3: Load Vision Embedding Model

In [3]:
# Load the vision embedding model
print("Loading vision embedding model...")
processor = AutoImageProcessor.from_pretrained("nomic-ai/nomic-embed-vision-v1.5", use_fast=True)
vision_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-vision-v1.5", trust_remote_code=True)

print("Model loaded successfully")
print(f"Processor: {type(processor)}")
print(f"Vision model: {type(vision_model)}")

Loading vision embedding model...


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Model loaded successfully
Processor: <class 'transformers.models.clip.image_processing_clip_fast.CLIPImageProcessorFast'>
Vision model: <class 'transformers_modules.nomic-ai.nomic-bert-2048.7710840340a098cfb869c4f65e87cf2b1b70caca.modeling_hf_nomic_bert.NomicVisionModel'>


## Step 4: Generate Embeddings for All Images

In [4]:
# Get list of all PNG files in the images folder
png_files = [f for f in os.listdir(images_dir) if f.lower().endswith('.png')]

# Initialize lists to store results
filenames = []
embeddings = []
images = []

print(f"Processing {len(png_files)} images for embeddings...")

# Loop through all PNG files
for i, filename in enumerate(png_files):
    try:
        # Load the image
        image_path = os.path.join(images_dir, filename)
        current_image = Image.open(image_path)
        images.append(np.asarray(current_image))

        # Apply the processing pipeline
        current_inputs = processor(current_image, return_tensors="pt")
        current_img_emb = vision_model(**current_inputs).last_hidden_state
        current_img_embeddings = F.normalize(current_img_emb[:, 0], p=2, dim=1)
        current_np_emb = current_img_embeddings.detach().numpy()[0]

        # Store results
        filenames.append(filename)
        embeddings.append(current_np_emb)

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(png_files)} images")

    except Exception as e:
        print(f"Error processing {filename}: {e}")

# Create DataFrame
df = pd.DataFrame({
    'filename': filenames,
    'embedding': embeddings
})

print(f"Successfully processed {len(df)} images")
display(df.head())

Processing 1000 images for embeddings...
Processed 100/1000 images
Processed 200/1000 images
Processed 300/1000 images
Processed 400/1000 images
Processed 500/1000 images
Processed 600/1000 images
Processed 700/1000 images
Processed 800/1000 images
Processed 900/1000 images
Processed 1000/1000 images
Successfully processed 1000 images


,filename,embedding
0,mnist_0000.png,"[-0.04062421, -0.026154635, -0.009534223, -0.0..."
1,mnist_0001.png,"[-0.05274934, -0.017808381, -0.009627939, -0.0..."
2,mnist_0002.png,"[-0.039479483, -0.020273585, -0.008993146, -0...."
3,mnist_0003.png,"[-0.041469358, -0.023677655, -0.014749996, -0...."
4,mnist_0004.png,"[-0.04619314, -0.028697776, -0.009102496, -0.0..."


## Step 5: Create 3D UMAP Visualization from Embeddings

In [5]:
# Convert embeddings list to numpy array matrix
embedding_matrix = np.stack(df['embedding'].values)

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print("Creating 3D UMAP coordinates...")

# Create 3D UMAP
umap_reducer = UMAP(n_components=3, random_state=42)
umap_coords_actual = umap_reducer.fit_transform(embedding_matrix)

# Add UMAP coordinates to dataframe
df['x'] = umap_coords_actual[:, 0]
df['y'] = umap_coords_actual[:, 1]
df['z'] = umap_coords_actual[:, 2]

print("UMAP coordinates created successfully")
print(f"X range: {df['x'].min():.2f} to {df['x'].max():.2f}")
print(f"Y range: {df['y'].min():.2f} to {df['y'].max():.2f}")
print(f"Z range: {df['z'].min():.2f} to {df['z'].max():.2f}")
display(df.head())

Embedding matrix shape: (1000, 768)
Creating 3D UMAP coordinates...


C:\Users\rober\miniforge3\envs\bob-env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP coordinates created successfully
X range: -0.10 to 12.73
Y range: 0.40 to 11.47
Z range: 2.11 to 10.34


,filename,embedding,x,y,z
0,mnist_0000.png,"[-0.04062421, -0.026154635, -0.009534223, -0.0...",5.827963,4.970609,6.495721
1,mnist_0001.png,"[-0.05274934, -0.017808381, -0.009627939, -0.0...",11.793096,2.019249,2.333435
2,mnist_0002.png,"[-0.039479483, -0.020273585, -0.008993146, -0....",4.324218,4.274768,6.286958
3,mnist_0003.png,"[-0.041469358, -0.023677655, -0.014749996, -0....",9.670835,1.456174,9.391963
4,mnist_0004.png,"[-0.04619314, -0.028697776, -0.009102496, -0.0...",3.871765,4.857382,4.566171


## Step 6: Save Results to CSV File

In [6]:
# Save the dataframe to CSV
output_path = base_dir / "data.csv"

# Convert embedding arrays to strings for CSV storage
df_to_save = df.copy()
df_to_save['embedding'] = df_to_save['embedding'].apply(lambda x: ','.join(map(str, x)))

df_to_save.to_csv(output_path, index=False)

print(f"Results saved to {output_path}")
print(f"Final dataframe shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df[['filename', 'x', 'y', 'z']].head(10))

Results saved to data.csv
Final dataframe shape: (1000, 5)
Columns: ['filename', 'embedding', 'x', 'y', 'z']


,filename,x,y,z
0,mnist_0000.png,5.827963,4.970609,6.495721
1,mnist_0001.png,11.793096,2.019249,2.333435
2,mnist_0002.png,4.324218,4.274768,6.286958
3,mnist_0003.png,9.670835,1.456174,9.391963
4,mnist_0004.png,3.871765,4.857382,4.566171
5,mnist_0005.png,5.441226,4.837626,6.394693
6,mnist_0006.png,9.084353,0.794404,8.810978
7,mnist_0007.png,9.839600,10.521388,7.439105
8,mnist_0008.png,10.034714,0.878803,9.904842
9,mnist_0009.png,5.158582,3.845076,6.762909
